**STEP 1: Set Up the Environment**

In [2]:
# Import required modules
import time
import multiprocessing
import cProfile

**STEP 2: Create a Sequential Program**


In [1]:
import time

# Define the compute-intensive function
def compute_sum(data):
    total = 0
    for x in data:
        total += x * x
    return total

# Generate a large dataset
data = list(range(1, 10_000_000))

# Measure execution time (Sequential Version)
start_time = time.time()

result = compute_sum(data)

end_time = time.time()
serial_time = end_time - start_time

print("Sequential Execution Time:", serial_time)

Sequential Execution Time: 1.0496196746826172


**STEP 3: Create a Parallel Version**

In [3]:
import time
import multiprocessing

# Compute-intensive function
def compute_sum(data):
    total = 0
    for x in data:
        total += x * x
    return total

# Split the dataset into chunks
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    return [data[i * chunk_size:(i + 1) * chunk_size] for i in range(num_chunks)]

# Parallel computation
def parallel_compute(data):
    num_processes = multiprocessing.cpu_count()
    chunks = chunk_data(data, num_processes)

    with multiprocessing.Pool(processes=num_processes) as pool:
        results = pool.map(compute_sum, chunks)

    return sum(results)

if __name__ == "__main__":
    # Generate dataset
    data = list(range(1, 10_000_000))

    # Measure execution time (Parallel Version)
    start_time = time.time()

    parallel_result = parallel_compute(data)

    end_time = time.time()
    parallel_time = end_time - start_time

    print("Parallel Execution Time:", parallel_time)

Parallel Execution Time: 2.467978000640869


**STEP 4: Calculate Speedup and Efficiency**

In [6]:
import multiprocessing

# Compute speedup
speedup = serial_time / parallel_time

# Compute efficiency
num_processors = multiprocessing.cpu_count()
efficiency = speedup / num_processors

print("Speedup:", speedup)
print("Efficiency:", efficiency)

# Pretty table output
print("\n" + "=" * 60)
print(f"{'VERSION':<15}{'EXECUTION TIME (seconds)':<25}")
print("=" * 60)
print(f"{'Sequential':<15}{serial_time:<25.6f}")
print(f"{'Parallel':<15}{parallel_time:<25.6f}")
print("=" * 60)

Speedup: 0.4252953933989924
Efficiency: 0.2126476966994962

VERSION        EXECUTION TIME (seconds) 
Sequential     1.049620                 
Parallel       2.467978                 


**STEP 5: Profile the Program (Identify Bottlenecks)**

*1. Profile the sequential version:*

In [8]:
import cProfile

cProfile.run("compute_sum(data)")

         425 function calls (419 primitive calls) in 0.695 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.502    0.502    0.502    0.502 60288059.py:5(compute_sum)
        2    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        4    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.182    0.182    0.684    0.684 <string>:1(<module>)
        1    0.000    0.000    0.011    0.011 asyncio.py:206(_handle_events)
        1    0.000    0.000    0.000    0.000 asyncio.py:231(add_callback)
        4    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        4    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 base_events.py:1907(_add_callback)
      4/3    0.000    0.000    0.695    0.232 base_events.py:1922(_run_once)
        2    0.000    0.000    0.000    0.000 b

*2. Observe:*

**1. Which functions take the most time?** *italicized text*
The function `compute_sum` takes the most time to execute, as shown by its highest total time (`tottime = 0.573 seconds`) and cumulative time (`cumtime = 0.573 seconds`) in the profiling results.

**2. Where might optimization help?**  
Optimization would be most effective inside the `compute_sum` function, particularly in the loop that iterates through the dataset and performs repeated multiplication and addition operations for each element.



*3.Write down one performance bottleneck you observe.* <br>
A key performance bottleneck is the sequential loop in `compute_sum`, which processes millions of elements one by one, making it computationally expensive and slow.

**STEP 6: Apply One Simple Optimization** (Option A: Reduce Loop Overhead)

In [9]:
import time

# Optimized function using built-in sum and generator expression
def compute_sum_optimized(data):
    return sum(x * x for x in data)

# Generate dataset
data = list(range(1, 10_000_000))

# Measure execution time (Optimized Sequential Version)
start_time = time.time()

optimized_result = compute_sum_optimized(data)

end_time = time.time()
optimized_time = end_time - start_time

print("Optimized Execution Time:", optimized_time)

Optimized Execution Time: 1.3979012966156006


**STEP 7: Re-Test Performance**



In [10]:
# Original function
def compute_sum(data):
    total = 0
    for x in data:
        total += x * x
    return total

# Optimized function
def compute_sum_optimized(data):
    return sum(x * x for x in data)

# Parallel helper functions
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    return [data[i * chunk_size:(i + 1) * chunk_size] for i in range(num_chunks)]

def parallel_compute(data):
    num_processes = multiprocessing.cpu_count()
    chunks = chunk_data(data, num_processes)

    with multiprocessing.Pool(processes=num_processes) as pool:
        results = pool.map(compute_sum, chunks)

    return sum(results)

if __name__ == "__main__":
    # Dataset
    data = list(range(1, 10_000_000))

    # --- Original Sequential ---
    start = time.time()
    compute_sum(data)
    serial_time = time.time() - start

    # --- Parallel ---
    start = time.time()
    parallel_compute(data)
    parallel_time = time.time() - start

    # --- Optimized Sequential ---
    start = time.time()
    compute_sum_optimized(data)
    optimized_time = time.time() - start

    # --- Results Table ---
    print("\n" + "=" * 65)
    print(f"{'VERSION':<25}{'EXECUTION TIME (seconds)':<30}")
    print("=" * 65)
    print(f"{'Original Sequential':<25}{serial_time:<30.6f}")
    print(f"{'Parallel':<25}{parallel_time:<30.6f}")
    print(f"{'Optimized Sequential':<25}{optimized_time:<30.6f}")
    print("=" * 65)


VERSION                  EXECUTION TIME (seconds)      
Original Sequential      0.673581                      
Parallel                 1.997312                      
Optimized Sequential     0.841096                      


**STEP 8: Short Analysis (Written Output)**

**1. Which version was fastest?**  
The original sequential version was the fastest, with an execution time of approximately 0.67 seconds. Surprisingly, both the parallel and optimized versions performed slower in this case.

**2. What was the speedup achieved?**  
The speedup was less than 1 since the parallel version (1.99 seconds) was slower than the original sequential version (0.67 seconds). This means there was no actual performance gain from parallelization.

**3. Was efficiency high or low? Why?**  
The efficiency was low because the overhead of creating multiple processes, splitting the data, and combining the results outweighed the benefits of parallel execution. This is common when the task is not heavy enough to justify multiprocessing overhead.

**4. What bottleneck did profiling reveal?**  
Profiling revealed that the main bottleneck was the loop inside the `compute_sum` function, where each element is processed sequentially, leading to high computation time.

**5. How did optimization affect performance?**  
The optimization slightly improved code readability but did not significantly improve performance. In fact, the optimized version (0.84 seconds) was still slower than the original sequential version, likely due to generator expression overhead.